## 1 卷积和池化层
### 1.1 理论计算题
输入一张大小为 $3 \times 32 \times 32$（通道数×高×宽）的彩色图像。通过一个卷积层，该层包含 16 个卷积核，每个卷积核的大小为 $3 \times 5 \times 5$。设定填充（Padding）为 2，步幅（Stride）为 2。
1. 请计算该卷积层输出的特征图（Feature Map）的尺寸（通道数×高×宽）。
2. 计算这个卷积操作中，单个输出通道的一个像素值，需要对输入进行多少次点乘（乘法）操作？


1. 假设输入图像尺寸为$n_h \times n_w$，卷积核尺寸为$k_h \times k_w$，填充为$p_h \times p_w$，步幅为$s_h \times s_w$，卷积输出尺寸计算公式为：

$$
H_{out} = \left\lfloor \frac{H_{in} - k_{h} + 2p_{h}}{s_{h}} \right\rfloor + 1
$$
$$
W_{out} = \left\lfloor \frac{W_{in} - k_{w} + 2p_{w}}{s_{w}} \right\rfloor + 1
$$

代入$n_h=32$，$n_w=32$，$k_h=5$，$k_w=5$，$p_h=2$，$p_w=2$，$s_h=2$，$s_w=2$，得到输出特征图的尺寸为：

$$
H_{out} = \left\lfloor \frac{32 - 5 + 2\times 2}{2} \right\rfloor + 1 = \left\lfloor \frac{31}{2} \right\rfloor + 1 = 16
$$
$$
W_{out} = \left\lfloor \frac{32 - 5 + 2\times 2}{2} \right\rfloor + 1 = 16
$$

又卷积核数量为16，所以输出特征图的通道数为16。因此，输出特征图的尺寸为$16 \times 16 \times 16$（通道数×高×宽）。

2. 单个输出通道的一个像素值，由输入对应区域与卷积核逐元素相乘后求和得到。
**乘法次数等于卷积核的总元素个数**，即$3 \times 5 \times 5 = 75$次。


### 1.2 编程题
不使用深度学习框架的底层 Pooling API（如 `torch.nn.MaxPool2d`），仅使用 Python 和 NumPy（或 PyTorch 基础张量操作），手动实现一个支持步幅（stride）和填充（padding）的二维最大池化（Max Pooling）前向传播函数。


In [3]:
import torch
import torch.nn.functional as F

def max_pool2d_manual(x, kernel_size, stride=1, padding=0):
    """
    手动实现二维最大池化前向传播
    参数:
        x: 输入张量，形状为 (batch_size, channels, height, width)
        kernel_size: 池化窗口大小，int或tuple (kh, kw)
        stride: 步幅，int或tuple (sh, sw)，默认1
        padding: 填充，int或tuple (ph, pw)，默认0
    返回:
        out: 池化后的输出张量
    """
    # 统一参数格式为tuple
    if isinstance(kernel_size, int):
        kh, kw = kernel_size, kernel_size
    else:
        kh, kw = kernel_size
    
    if isinstance(stride, int):
        sh, sw = stride, stride
    else:
        sh, sw = stride
    
    if isinstance(padding, int):
        ph, pw = padding, padding
    else:
        ph, pw = padding
    
    batch_size, channels, H_in, W_in = x.shape
    
    # 计算输出尺寸
    H_out = (H_in + 2*ph - kh) // sh + 1
    W_out = (W_in + 2*pw - kw) // sw + 1
    
    # 对输入进行填充，填充值为负无穷
    x_pad = F.pad(x, (pw, pw, ph, ph), value=float('-inf'))
    # 展开成 (N, C, kh*kw, H_out*W_out)
    unfolded = F.unfold(x_pad, kernel_size=(kh, kw), stride=(sh, sw))
    # 在通道维度取最大值
    out = unfolded.max(dim=2)[0].view(batch_size, channels, H_out, W_out)
    
    return out


## 2 LeNet, AlexNet, VGG 和 NiN
### 2.1 理论计算题
在 VGG 网络中，作者频繁使用多个 $3 \times 3$ 卷积核级联来代替较大的卷积核（如 $5 \times 5$ 或 $7 \times 7$）。假设输入和输出的特征图通道数均为 $C$。
1. 计算一个 $5 \times 5$ 卷积层（不带偏置）的参数量。
2. 计算两个串联的 $3 \times 3$ 卷积层（不带偏置，两层通道数都为 $C$）的总参数量。


不带偏置的卷积层参数量 = $C_{in} \times C_{out} \times k_{h} \times k_{w}$    
1. $5 \times 5$ 卷积层的参数量为 $C \times C \times 5 \times 5 = 25C^2$
2. 两个 $3 \times 3$ 卷积层的总参数量为 $C \times C \times 3 \times 3 + C \times C \times 3 \times 3 = 18C^2$   
VGG通过这种设计在**保持相同感受野**的同时，**减少参数**并**增加非线性**，提升模型表达能力。


### 2.2 编程题
NiN 网络的核心创新是引入了“1x1 卷积”组成的 NiN 块来代替传统的全连接层，以减少参数量。请使用 PyTorch（`torch.nn.Sequential`）定义一个标准的 NiN 块（NiN Block）。

要求：NiN 块接收输入通道数 `in_channels` 和输出通道数 `out_channels`，它由一个普通的卷积层（指定窗口大小 `kernel_size`，步幅 `stride`，填充 `padding`）以及两个随后的 $1 \times 1$ 卷积层级联组成。每层卷积后都需要紧跟一个 ReLU 激活层。


In [4]:
import torch
import torch.nn as nn

def nin_block(in_channels, out_channels, kernel_size, stride, padding):
    """
    实现标准NiN块
    参数:
        in_channels: 输入通道数
        out_channels: 输出通道数
        kernel_size: 第一个卷积层的核大小
        stride: 第一个卷积层的步幅
        padding: 第一个卷积层的填充
    返回:
        nn.Sequential: NiN块
    """
    return nn.Sequential(
        nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(inplace=True),
        nn.Conv2d(out_channels, out_channels, kernel_size=1),
        nn.ReLU(inplace=True)
    )

# 测试代码
if __name__ == "__main__":
    block = nin_block(in_channels=3, out_channels=16, kernel_size=5, stride=1, padding=2)
    x = torch.randn(1, 3, 32, 32)
    out = block(x)
    print("NiN块输出形状:", out.shape)  

NiN块输出形状: torch.Size([1, 16, 32, 32])



## 3 Inception, 批量归一化和残差网络
### 3.1 理论计算题
在一个小批量（Mini-batch）训练中，某一个通道内某一特定空间位置的特征值在 4 个样本上的输出分别为：$x_{1}=2$，$x_{2}=4$，$x_{3}=6$，$x_{4}=8$。假设当前批量归一化层学到的缩放参数 $\gamma=2$，平移参数 $\beta=1$，常数 $\epsilon=0$。

请计算这 4 个样本经由该 Batch Normalization 层转化后的最终输出值 $y_{1}$，$y_{2}$，$y_{3}$，$y_{4}$。      



从形式上来说，用 $x \in \mathcal{B}$表示一个来自小批量的输入，批量规范化BN根据以下表达式转换：     
$$\mathrm{BN}(\mathbf{x}) = \gamma \odot \frac{\mathbf{x} - \hat{\mu}_{\mathrm{B}}}{\hat{\sigma}_{\mathrm{B}}} + \beta.$$
其中，$\hat{\mu}_B$ 和 $\hat{\sigma}^2_B$ 分别是小批量 $\mathcal{B}$ 的均值和方差，计算公式如下：
$$\hat{\mu}_B = \frac{1}{|B|} \sum_{x \in B} x, \quad \hat{\sigma}^2_B = \frac{1}{|B|} \sum_{x \in B} (x - \hat{\mu}_B)^2 + \epsilon.$$
$$\hat{x}_i = \frac{x_i - \hat{\mu}_B}{\sqrt{\hat{\sigma}^2_B+\epsilon}}, y_i = \gamma \hat{x}_i + \beta.$$
首先计算小批量的均值和方差：
$$\hat{\mu}_B = \frac{2 + 4 + 6 + 8}{4} = 5.$$
$$\hat{\sigma}^2_B = \frac{(2-5)^2 + (4-5)^2 + (6-5)^2 + (8-5)^2}{4} + 0 = \frac{9 + 1 + 1 + 9}{4} = 5.$$
然后计算每个样本经过批量归一化后的输出：
$$y_{i} = 2 \cdot \frac{x_{i} - 5}{\sqrt{5}} + 1.$$
因此，最终输出值为：
$$y_{1} = 2 \cdot \frac{2 - 5}{\sqrt{5}} + 1 = 1 - \frac{6}{\sqrt{5}} \approx -1.683,$$
$$y_{2} = 2 \cdot \frac{4 - 5}{\sqrt{5}} + 1 = 1 - \frac{2}{\sqrt{5}} \approx 0.106,$$
$$y_{3} = 2 \cdot \frac{6 - 5}{\sqrt{5}} + 1 = 1 + \frac{2}{\sqrt{5}} \approx 1.894,$$
$$y_{4} = 2 \cdot \frac{8 - 5}{\sqrt{5}} + 1 = 1 + \frac{6}{\sqrt{5}} \approx 3.683$$



- $\epsilon$的作用：防止方差为0时除零错误，通常取$\epsilon=10-04$。   
- 训练时用当前batch统计量，测试时用滑动平均统计量   
- $\gamma$和$\beta$是可学习参数，恢复表达能力



### 3.2 编程题
残差网络（ResNet）通过引入跨层连接（残差连接）解决了深层网络的梯度消失问题。请用 PyTorch 自定义一个残差块类 `Residual`。

要求：该块包含两个具有相同输出通道数的 $3 \times 3$ 卷积层，每个卷积层后跟一个批量归一化层。如果 `use_1x1conv=True`，则需要对输入应用一个 $1 \times 1$ 的卷积层来调整输入的通道数和形状，以便它能和第二层卷积的输出进行按元素相加（$f(x)+x$）。


In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class Residual(nn.Module):
    """
    实现ResNet标准残差块
    """
    def __init__(self, in_channels, out_channels, stride=1, use_1x1conv=False):
        super().__init__()
        # 主路径两个3×3卷积层
        self.main_path = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, stride, 1),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, 1, 1),
            nn.BatchNorm2d(out_channels)
        )
        self.shortcut = nn.Sequential()
        if use_1x1conv:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, stride),
                nn.BatchNorm2d(out_channels)
            )
        
    def forward(self, x):
        return F.relu(self.main_path(x) + self.shortcut(x))

# 测试代码
if __name__ == "__main__":
    # 测试通道数不变、尺寸不变的情况
    blk1 = Residual(3, 3)
    x = torch.randn(1, 3, 32, 32)
    print("残差块1输出形状:", blk1(x).shape)  # torch.Size([1, 3, 32, 32])
    
    # 测试通道数增加、尺寸减半的情况
    blk2 = Residual(3, 6, stride=2, use_1x1conv=True)
    print("残差块2输出形状:", blk2(x).shape)  # torch.Size([1, 6, 16, 16])

残差块1输出形状: torch.Size([1, 3, 32, 32])
残差块2输出形状: torch.Size([1, 6, 16, 16])



## 4 图像增广, 微调和样式迁移
### 4.1 理论计算题
在微调（Fine-tuning）任务中，我们通常会在一个大型源数据集（如 ImageNet）上预训练好的网络模型基础上，去适应一个新的目标数据集。请回答以下关于微调理论的问题：
1. 为什么我们通常对除了最终输出层之外的“底层特征提取层”设置较小的学习率（甚至将其参数固定/冻结），而对新初始化的“顶层输出层”设置较大的学习率？
2. 如果目标数据集非常小，且与源数据集非常相似，我们应该采取什么样的微调策略以防止过拟合？



1. **底层特征提取层**通常学习到的是通用的、基础的特征（*边缘、纹理、形状等*），这些特征在不同的任务之间具有较高的迁移性（*已经训练得非常充分，且适用于大多数视觉任务*）。因此，我们希望保留这些已经学习到的有用特征，而不对其进行大的修改。较大的学习率可能会破坏这些已学习的特征，导致性能下降。   
**顶层输出层**是针对特定任务设计的，与新目标数据集的类别完全不同且强关联。该层参数是**随机初始化**的，需要根据新的数据集进行调整，因此设置较大的学习率。
2. 如果目标数据集非常小，且与源数据集非常相似，我们应该**最小化对预训练知识的破坏，同时最大化利用目标数据集进行训练**：
   - 冻结大部分底层特征提取层的参数，只训练顶层输出层。
   - 渐进式解冻底层特征提取层，先用较小的学习率更新底层特征提取层的参数，再用较大的学习率更新顶层输出层的参数。
   - 低学习率微调，先在小数据集上进行微调，再在大数据集上进行微调，以避免过拟合数据。
   - 在训练过程中使用正则化技术，如Dropout或L2正则化。
   - 知识蒸馏技术，将预训练模型的输出作为教师模型，将新模型的输出作为学生模型，通过训练学生模型来学习教师模型的知识。


### 4.2 编程题
图像增广能有效增强模型的泛化能力。请利用 `torchvision.transforms` 模块创建一个组合图像增广管道（Pipeline）。
1. 随机对图像进行裁剪，使其面积比例在 0.08 到 1.0 之间，并将裁剪后的图像缩放到 $224 \times 224$ 像素。
2. 拥有 50% 的概率对图像进行水平翻转。
3. 随机改变图像的亮度（Brightness）、对比度（Contrast）和饱和度（Saturation），变化范围设为 0.5。
4. 最终将图像转换为 PyTorch 张量（Tensor）。


In [6]:
from torchvision import transforms

def get_augmentation_pipeline():
    """
    创建题目要求的图像增广管道
    """
    return transforms.Compose([
        # 1. 随机裁剪并缩放到224×224，面积比例0.08-1.0
        transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
        # 2. 50%概率水平翻转
        transforms.RandomHorizontalFlip(p=0.5),
        # 3. 随机改变亮度、对比度、饱和度，变化范围0.5
        transforms.ColorJitter(brightness=0.5, contrast=0.5, saturation=0.5),
        # 4. 转换为PyTorch张量
        transforms.ToTensor()
    ])

# 测试代码
if __name__ == "__main__":
    from PIL import Image
    import numpy as np
    
    # 创建测试图像
    img = Image.fromarray(np.random.randint(0, 255, (500, 500, 3), dtype=np.uint8))
    aug = get_augmentation_pipeline()
    out = aug(img)
    print("增广后张量形状:", out.shape)  # torch.Size([3, 224, 224])

增广后张量形状: torch.Size([3, 224, 224])



## 5 目标检测, 计算机视觉训练技巧
### 5.1 理论计算题
在目标检测中，交并比（IoU）用于衡量预测边界框与真实边界框的重合程度。已知图像中两个边界框（以 `[左上角 x, 左上角 y, 右下角 x, 右下角 y]` 格式表示）：
1. 真实框（Ground Truth）$A=[10,10,50,50]$
2. 预测框（Prediction Box）$B=[30,30,70,70]$

请计算边界框 A 和边界框 B 之间的 IoU 准确值。



IoU（Intersection over Union）计算公式为：
$$IoU = \frac{|A \cap B|}{|A \cup B|}$$
首先计算两个边界框的交集（Intersection）和并集（Union）：
- 边界框 A 的左上角坐标为 (10, 10)，右下角坐标为 (50, 50)，宽度和高度均为 40。
- 边界框 B 的左上角坐标为 (30, 30)，右下角坐标为 (70, 70)，宽度和高度均为 40。      
交集部分的左上角坐标为 (30, 30)，右下角坐标为 (50, 50)，宽度和高度均为 20，因此交集面积为 $20 \times 20 = 400$。      
并集面积可以通过以下方式计算：
$$|A \cup B| = |A| + |B| - |A \cap B| = 40 \times 40 + 40 \times 40 - 400 = 2800.$$
因此，IoU 的准确值为：
$$IoU = \frac{400}{2800} = \frac{1}{7} \approx 0.1429.$$



### 5.2 编程题
在计算机视觉训练技巧中，标签平滑（Label Smoothing）通过防止模型过于自信地预测某些类别来提高泛化性。标准交叉熵使用独热编码（One-hot），若设置平滑因子 $\epsilon=0.1$，则对于 K 分类问题，真实标签对应的目标概率从 1 变为 $1-\epsilon$，其余错误类别的概率从 0 变为 $\frac{\epsilon}{K-1}$。

请实现一个计算标签平滑后交叉熵损失的函数。

In [7]:
import torch
import torch.nn.functional as F

def label_smoothing_cross_entropy(logits, targets, epsilon=0.1):
    """
    手动实现标签平滑后的交叉熵损失
    参数:
        logits: 模型输出，未经过softmax，形状为 (batch_size, num_classes)
        targets: 真实标签，形状为 (batch_size,)
        epsilon: 平滑因子，默认0.1
    返回:
        loss: 平均损失值
    """
    batch_size = logits.shape[0]
    # 计算log_softmax，数值更稳定
    log_probs = F.log_softmax(logits, dim=1)
    # 创建平滑后的目标分布
    smooth_targets = torch.full_like(log_probs, epsilon / (log_probs.shape[1] - 1))
    smooth_targets.scatter_(1, targets.unsqueeze(1), 1 - epsilon)
    
    return -(log_probs * smooth_targets).sum(dim=1).mean()